In [10]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# Configuration for small sample
NUM_USERS = 10
RECORDS_PER_ACTIVITY = 20
ACTIVITIES = ['DLI training', 'downloads', 'webinars', 'whitepapers', 'events', 'trials', 'support']

# Regional US Bounds for GPS Coordinates
REGIONS = {
    'East': {'lat': (35.0, 45.0), 'lon': (-80.0, -70.0)},
    'West': {'lat': (32.0, 48.0), 'lon': (-125.0, -114.0)},
    'South': {'lat': (25.0, 35.0), 'lon': (-100.0, -80.0)},
    'Central': {'lat': (35.0, 48.0), 'lon': (-105.0, -90.0)}
}

def generate_final_schema(num_users, records_per_act):
    # 1. Central Contact & Account Data
    user_ids = [str(uuid.uuid4())[:8] for _ in range(num_users)]
    assigned_regions = np.random.choice(list(REGIONS.keys()), size=num_users)
    
    contacts = pd.DataFrame({
        'user_id': user_ids,
        'account_id': [f"ACC-{np.random.randint(1000, 9999)}" for _ in range(num_users)],
        'region': assigned_regions,
        'segment': np.random.choice(['Enterprise', 'SMB', 'Public Sector'], size=num_users),
        'base_lat': [np.random.uniform(REGIONS[r]['lat'][0], REGIONS[r]['lat'][1]) for r in assigned_regions],
        'base_lon': [np.random.uniform(REGIONS[r]['lon'][0], REGIONS[r]['lon'][1]) for r in assigned_regions]
    })

    all_data = []

    # 2. Activity and Metadata Pairs
    for act in ACTIVITIES:
        # Engagement Metadata
        eng_ids = [f"{act[:3].upper()}-{i:03}" for i in range(20)]
        meta = pd.DataFrame({
            'engagement_id': eng_ids,
            'engage_name': [f"{act.capitalize()} Session {i}" for i in range(20)],
            'category': act
        })

        # Activity Records
        act_df = pd.DataFrame({
            'activity_id': [f"ACT-{uuid.uuid4().hex[:6]}" for _ in range(records_per_act)],
            'user_id': np.random.choice(user_ids, size=records_per_act),
            'engagement_id': np.random.choice(eng_ids, size=records_per_act),
            'timestamp': [datetime(2026, 1, 1) + timedelta(days=np.random.randint(0, 60)) for _ in range(records_per_act)],
            'duration_minutes': np.random.randint(5, 120, size=records_per_act)
        })

        # 3. Join and Apply GPS Coordinates
        joined = act_df.merge(meta, on='engagement_id').merge(contacts, on='user_id')
        
        # Add GPS Coordinates with Jitter (to ensure non-overlapping locations)
        joined['latitude'] = joined['base_lat'] + np.random.uniform(-0.05, 0.05, size=len(joined))
        joined['longitude'] = joined['base_lon'] + np.random.uniform(-0.05, 0.05, size=len(joined))

        # Filter and Order specifically to your request
        final_cols = [
            'activity_id', 'user_id', 'engagement_id', 'timestamp', 
            'duration_minutes', 'engage_name', 'category', 
            'account_id', 'region', 'segment', 'latitude', 'longitude'
        ]
        all_data.append(joined[final_cols])

    return pd.concat(all_data, ignore_index=True)

# Generate sample
df_sample = generate_final_schema(NUM_USERS, RECORDS_PER_ACTIVITY)
print(df_sample.head())

  activity_id   user_id engagement_id  timestamp  duration_minutes  \
0  ACT-bbc254  244e8d95       DLI-012 2026-02-14                16   
1  ACT-60066c  244e8d95       DLI-003 2026-02-10                16   
2  ACT-83bbbf  f2a8d369       DLI-003 2026-01-24                57   
3  ACT-0bf5f4  244e8d95       DLI-000 2026-01-21                73   
4  ACT-a47389  82a5d70a       DLI-016 2026-01-16                34   

               engage_name      category account_id   region        segment  \
0  Dli training Session 12  DLI training   ACC-9477  Central            SMB   
1   Dli training Session 3  DLI training   ACC-9477  Central            SMB   
2   Dli training Session 3  DLI training   ACC-6084     East  Public Sector   
3   Dli training Session 0  DLI training   ACC-9477  Central            SMB   
4  Dli training Session 16  DLI training   ACC-7977     West     Enterprise   

    latitude   longitude  
0  38.568701  -95.527918  
1  38.490575  -95.493740  
2  42.050244  -72.47510

In [12]:
birds_eye_view.to_csv(r"C:\Users\mikha\Customers\Nvidia\activity_data.csv", index=False)
